In [21]:
import pandas as pd
import os
import cv2
import random
import shutil
import numpy as np
import tensorflow as tf


In [22]:
# -- Configuration --
csv_path = '/workspace/small_dataset/labels.csv'
source_images_dir = '/workspace/small_dataset/'
output_dir = 'yolo_dataset'

TARGET_CLASSES = ['basophil', 'eosinophil', 'seg_neutrophil', 'lymphocyte', 'monocyte']
MAX_ORIGINALS_PER_CLASS = 250

# -- Set Seed
def set_seed(seed_value=12345):
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

set_seed()

In [23]:
# 1. Load the data
df = pd.read_csv(csv_path)
# print(df.head(5))
# -- Debug --
# Print all classes
all_lables = df['label'].unique().tolist()
print(all_lables)

from IPython.display import Image, display
# -- Debug -- 
# Show images.
# CLASS_TO_SEE = ['basophil']
# showcase_df = df[df['label'].isin(CLASS_TO_SEE)]
# show_first_10_images = showcase_df[:10]
# for img_num in first_10_smudge_images['img_num']:
#     BboxParamsimg_path = os.path.join(source_images_dir, str(img_num) + '_7.jpg')
#     print(img_path)
#     display(Image(filename=img_path))

# 2. Filter out unwanted classes
df = df[df['label'].isin(TARGET_CLASSES)]

# 3. Map string labels to integer class IDs
unique_labels = df['label'].unique().tolist()
label_to_id = {label: idx for idx, label in enumerate(unique_labels)}
print(label_to_id)

['metamyelocyte', 'myelocyte', 'seg_neutrophil', 'band_neutrophil', 'lymphocyte', 'monocyte', 'immature_wbc', 'eosinophil', 'basophil', 'abnormal_lymphocyte', 'smudge', 'blast', 'promyelocyte']
{'seg_neutrophil': 0, 'lymphocyte': 1, 'monocyte': 2, 'eosinophil': 3, 'basophil': 4}


In [24]:
# -- Debug -- 
# Chekc bounding box.

# Pick a row.
# row = df.iloc[2]

# # Extract the data from the row
# img_id = int(row['img_num'])
# img_path = os.path.join(source_images_dir, str(img_id) + '_7.jpg')
# x = int(row['rel_x'])
# y = int(row['rel_y'])
# w = int(row['cell_roi_w'])
# h = int(row['cell_roi_h'])
# label = row['label']

# image = cv2.imread(img_path)
# image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# top_left = (x, y)
# bottom_right = (x + w, y + h)
# box_color = (255, 0, 0)
# thickness = 2

# cv2.rectangle(image, top_left, bottom_right, box_color, thickness)

# font = cv2.FONT_HERSHEY_SIMPLEX
# font_scale = 0.5
# text_color = (255, 0, 0) # Red text
    
# # (x, y - 5) pushes the text slightly above the top line of the box
# cv2.putText(image, label, (x, y - 5), font, font_scale, text_color, thickness)

# plt.figure(figsize=(8,8))
# plt.imshow(image)
# plt.show()

In [25]:
# 4. Pre-augment selection.
selected_img_ids = set()
class_counts = {c: 0 for c in TARGET_CLASSES}
127
all_image_ids = df['img_num'].unique().tolist()
random.seed(42) 
random.shuffle(all_image_ids) 

for img_id in all_image_ids:
    img_classes = df[df['img_num'] == img_id]['label'].unique()
    needs_image = any(class_counts[c] < MAX_ORIGINALS_PER_CLASS for c in img_classes)
    
    if needs_image:
        selected_img_ids.add(img_id)
        for c in img_classes:
            class_counts[c] += 1
            
    if all(count >= MAX_ORIGINALS_PER_CLASS for count in class_counts.values()):
        break

print(f"\nBase Selection (Before Augmentation):")
for c, count in class_counts.items():
    print(f" - {c}: {count}")


Base Selection (Before Augmentation):
 - basophil: 127
 - eosinophil: 250
 - seg_neutrophil: 250
 - lymphocyte: 250
 - monocyte: 250


In [26]:

df = df[df['img_num'].isin(selected_img_ids)]

# --- 4. Shuffle and Split ---
image_ids = list(selected_img_ids)
random.shuffle(image_ids)

train_idx = int(len(image_ids) * 0.7)
valid_idx = int(len(image_ids) * 0.9)

train_ids = set(image_ids[:train_idx])
valid_ids = set(image_ids[train_idx:valid_idx])
test_ids  = set(image_ids[valid_idx:])

def get_split(img_id):
    if img_id in train_ids: return "train"
    elif img_id in valid_ids: return "valid"
    else: return "test"

splits = ["train", "valid", "test"]
for split in splits:
    os.makedirs(os.path.join(output_dir, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, "labels"), exist_ok=True)

valid_extensions = ('.jpg', '.jpeg', '.png')
all_source_files = [f for f in os.listdir(source_images_dir) if f.lower().endswith(valid_extensions)]

In [27]:
# 5. Augmentation (Optional)

import albumentations as alb
from matplotlib import pyplot as plt

# 4.1 Convert csv relative bounding box
# to standard YOLO

# 4.2 Define augmentation pipeline
augment_pipeline = alb.Compose([
    alb.HorizontalFlip(p=0.5), # 50% chance to flip
    alb.VerticalFlip(p=0.5),
], bbox_params=alb.BboxParams(format='yolo',
                              label_fields=['class_ids']))


AUGMENT_TARGETS = {
    'basophil': 150,      # Generate augmented basophils until we hit n total
}

# The maximum number of fake variations to create from a single original image. 
# (Increase this if your target is very high and original image count is very low)
MAX_AUGS_PER_IMAGE = 5 

In [28]:
# --- 5. Process, Move, and Selectively Augment ---
saved_counts = {c: 0 for c in TARGET_CLASSES}
for img_num, group in df.groupby("img_num"):
    
    matched_files = [
        f for f in all_source_files 
        if f.startswith(f"{img_num}.") or f.startswith(f"{img_num}_")
    ]
    
    if not matched_files:
        continue
        
    split_name = get_split(img_num)
    
    for filename in matched_files:
        src_img_path = os.path.join(source_images_dir, filename)
        base_name = os.path.splitext(filename)[0] 
        
        dst_img_path = os.path.join(output_dir, split_name, "images", filename)
        dst_label_path = os.path.join(output_dir, split_name, "labels", f"{base_name}.txt")
        
        img = cv2.imread(src_img_path)
        if img is None:
            continue
        img_height, img_width = img.shape[:2]

        # -- Important --
        bboxes = []
        class_ids = []
        class_names = []
        
        # 5A. Gather coordinates 
        for _, row in group.iterrows():
            c_name = row['label']
            c_id = label_to_id[c_name]

            
            
            xmin = row['rel_x']
            ymin = row['rel_y']
            box_width = row['cell_roi_w'] 
            box_height = row['cell_roi_h'] 
            
            # -- Debug -- 
            # Check the normal images.
            # Use absolute.
            # color=(0, 255, 0)
            # thickness=2
            # cv2.rectangle(img, (norm_x, norm_y), (norm_x + norm_w, norm_y + norm_h), color, thickness)

            # plt.figure(figsize=(8,8))
            # plt.imshow(img)
            # plt.show()

        
            x_center = xmin + (box_width / 2.0)
            y_center = ymin + (box_height / 2.0)
            
            norm_x = max(0.0, min(1.0, x_center / img_width))
            norm_y = max(0.0, min(1.0, y_center / img_height))
            norm_w = max(0.0, min(1.0, box_width / img_width))
            norm_h = max(0.0, min(1.0, box_height / img_height))


            
            bboxes.append([norm_x, norm_y, norm_w, norm_h])
            class_ids.append(c_id)
            class_names.append(c_name)

        # 5B. Save the Original
        shutil.copy(src_img_path, dst_img_path)
        with open(dst_label_path, "w") as f:
            for bbox, c_id in zip(bboxes, class_ids):
                f.write(f"{c_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
        
        for c_name in class_names:
            saved_counts[c_name] += 1

        # --- 5C. AUGMENT BASED ON EXPLICIT TARGETS ---
        if split_name == "train":
            aug_count = 0
            
            # Condition: If ANY class in this image is in AUGMENT_TARGETS AND hasn't reached its goal yet
            while any(c in AUGMENT_TARGETS and saved_counts[c] < AUGMENT_TARGETS[c] for c in class_names) and aug_count < MAX_AUGS_PER_IMAGE:
                
                transformed = augment_pipeline(image=img, bboxes=bboxes, class_ids=class_ids)
                aug_img = transformed['image']
                
                aug_bboxes = transformed['bboxes']
                aug_class_ids = transformed['class_ids']
                
                aug_idx = random.randint(10000, 99999) 
                aug_img_name = f"{base_name}_aug{aug_idx}.jpg"
                aug_txt_name = f"{base_name}_aug{aug_idx}.txt"
                
                aug_img_path = os.path.join(output_dir, split_name, "images", aug_img_name)
                aug_txt_path = os.path.join(output_dir, split_name, "labels", aug_txt_name)
                
                cv2.imwrite(aug_img_path, aug_img)
                with open(aug_txt_path, "w") as f:
                    for bbox, c_id in zip(aug_bboxes, aug_class_ids):
                        f.write(f"{c_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")


                # -- Debug -- 
                # Check the augmented images.
                # x = int(aug_bboxes[0][0])
                # y = int(aug_bboxes[0][1])
                # w = int(aug_bboxes[0][2])
                # h = int(aug_bboxes[0][3])
                # color=(0, 255, 0)
                # thickness=2
                # cv2.rectangle(aug_img, (x, y), (x + w, y + h), color, thickness)

                # plt.figure(figsize=(8,8))
                # plt.imshow(aug_img)
                # plt.show()
                
                for c_name in class_names:
                    saved_counts[c_name] += 1
                    
                aug_count += 1


In [29]:
saved_counts

{'basophil': 243,
 'eosinophil': 250,
 'seg_neutrophil': 250,
 'lymphocyte': 250,
 'monocyte': 250}

In [30]:
import yaml
# --- 6. Generate data.yaml ---
yaml_content = {
    "path": os.path.abspath(output_dir), 
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {idx: label for idx, label in enumerate(unique_labels)}
}

yaml_path = os.path.join(output_dir, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"\nSuccess! Built split YOLO dataset in '{output_dir}/'")
print(f"Created Train: {len(train_ids)} | Valid: {len(valid_ids)} | Test: {len(test_ids)} (Base IDs)")

print(f"\nFinal Class Counts (Originals + Augmentations):")
for c, count in saved_counts.items():
    print(f" - {c}: {count}")


Success! Built split YOLO dataset in 'yolo_dataset/'
Created Train: 788 | Valid: 226 | Test: 113 (Base IDs)

Final Class Counts (Originals + Augmentations):
 - basophil: 243
 - eosinophil: 250
 - seg_neutrophil: 250
 - lymphocyte: 250
 - monocyte: 250
